# WSF Tracker — Year-Code Sensitivity Analysis

**Purpose:** Three analyses of how WSF Tracker temporal choices affect building-dataset validation accuracy:

1. **Section 1 — Sensitivity sweep:** re-run WSF raster validation across a range of `as_of_code` cutoffs (2019 → mid-2025) to quantify how sensitive accuracy metrics are to the chosen year.
2. **Section 2 — Urban growth rate vs. F1:** test whether cities with faster WSF urban growth between the reference image year and mid-2025 show systematically lower vector F1 scores.
3. **Section 3 — Temporally-aligned validation:** re-run WSF raster validation for each city using the cutoff closest to that city's reference imagery date, giving a fairer per-city accuracy estimate free of temporal mismatch.

**WSF year-code encoding (bi-annual):**
`code = round((year − 2016) × 2)`, range [1, 20].
Code 1 = mid-2016, code 2 ≈ end-2016, …, code 19 ≈ mid-2025, code 20 ≈ end-2025.

**Outputs:**
- `outputs/wsf_year_sensitivity.csv` — per-city metrics for each year cutoff (Section 1)
- `outputs/figures/growth_rate_vs_f1.png` — scatter plot (Section 2)
- `outputs/wsf_growth_vs_f1_correlations.csv` — correlation table (Section 2)
- `outputs/wsf_aligned_validation.csv` — per-city temporally-aligned results (Section 3)

Created by: Caroline Gevaert — The World Bank
Financed by: The Gates Foundation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.windows import from_bounds, Window
from rasterio.transform import Affine
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling
import yaml
import warnings

# Suppress Colab/Jupyter's own utcnow deprecation warning — it comes from
# jupyter_client internals and fires on every kernel message; not actionable.
warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow",
    category=DeprecationWarning,
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation"
)
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation


In [3]:
# WSF Tracker bi-annual encoding: code = round((year - 2016) * 2), range [1, 20].
# Odd codes = mid-year; even codes = year-end / next-year-start.
YEAR_CODE_MAP = {
     1: 2016.5,  2: 2017.0,  3: 2017.5,  4: 2018.0,  5: 2018.5,
     6: 2019.0,  7: 2019.5,  8: 2020.0,  9: 2020.5, 10: 2021.0,
    11: 2021.5, 12: 2022.0, 13: 2022.5, 14: 2023.0, 15: 2023.5,
    16: 2024.0, 17: 2024.5, 18: 2025.0, 19: 2025.5, 20: 2026.0,
}

code_to_year = YEAR_CODE_MAP
year_to_code = {v: k for k, v in code_to_year.items()}

BASELINE_CODE = 19  # current config value in validation_configs.yaml (≈ mid-2025)

# Sweep: even codes (integer years 2019–2026) plus the config baseline (code 19)
# so the baseline always appears in the sensitivity summary table.
START_YEAR = 2019
END_YEAR   = max(code_to_year.values())   # 2026.0

year_sweep_base = [y for y in range(int(START_YEAR), int(END_YEAR) + 1) if y in year_to_code]
SWEEP_CODES = sorted(set([year_to_code[y] for y in year_sweep_base] + [BASELINE_CODE]))
SWEEP_YEARS = [code_to_year[c] for c in SWEEP_CODES]

print(f"Encoding: codes 1–{max(code_to_year.keys())}, "
      f"years {min(code_to_year.values())}–{max(code_to_year.values())}")
print(f"Baseline: as_of_code={BASELINE_CODE} → {code_to_year[BASELINE_CODE]} (mid-2025)")
print(f"\nSweep ({len(SWEEP_CODES)} cutoffs):")
for code, year in zip(SWEEP_CODES, SWEEP_YEARS):
    marker = "  ← config baseline" if code == BASELINE_CODE else ""
    print(f"  as_of_code={code:3d}  →  {year}{marker}")

Encoding: codes 1–20, years 2016.5–2026.0
Baseline: as_of_code=19 → 2025.5 (mid-2025)

Sweep (9 cutoffs):
  as_of_code=  6  →  2019.0
  as_of_code=  8  →  2020.0
  as_of_code= 10  →  2021.0
  as_of_code= 12  →  2022.0
  as_of_code= 14  →  2023.0
  as_of_code= 16  →  2024.0
  as_of_code= 18  →  2025.0
  as_of_code= 19  →  2025.5  ← config baseline
  as_of_code= 20  →  2026.0


## Section 1 — Year-code sensitivity sweep

For each `as_of_code` in the sweep, re-runs the WSF raster validation across all cities in the AOI tracker and records per-city accuracy metrics. The summary table flags any cutoff that differs from the current pipeline baseline (`as_of_code: 19`) by more than 0.05 F1.

In [4]:
# ---- Shared helpers ----

def _pixel_area_from_transform(transform) -> float:
    return float(abs(transform.a * transform.e))

def open_in_target_crs(src, target_crs: str):
    if src.crs is None:
        raise ValueError("Raster has no CRS.")
    if str(src.crs) == str(target_crs):
        return src
    return WarpedVRT(src, crs=target_crs, resampling=Resampling.nearest)

def _load_multi_file(base_dir: Path, file_spec: str, crs: str):
    """
    Load one or more pipe-separated vector files from base_dir.
    Returns (GeoDataFrame, list_of_missing_filenames).
    GeoDataFrame is None when no file could be found.
    """
    names = [n.strip() for n in str(file_spec).split("|") if n.strip()]
    parts, missing = [], []
    for name in names:
        p = base_dir / name
        if p.exists():
            parts.append(gpd.read_file(p).to_crs(crs))
        else:
            missing.append(name)
    if not parts:
        return None, missing
    gdf = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
    return gdf, missing

def _read_tile(ds, win, fill):
    """Read a window from ds, working around WarpedVRT's boundless=True restriction."""
    if not isinstance(ds, WarpedVRT):
        return ds.read(1, window=win, boundless=True, fill_value=fill)
    # WarpedVRT forbids boundless reads — clamp to valid extent and pad the rest.
    out_h = max(1, int(round(win.height)))
    out_w = max(1, int(round(win.width)))
    arr   = np.full((out_h, out_w), fill, dtype=ds.dtypes[0])
    r0 = max(0, int(win.row_off))
    c0 = max(0, int(win.col_off))
    r1 = min(ds.height, int(round(win.row_off + win.height)))
    c1 = min(ds.width,  int(round(win.col_off + win.width)))
    if r1 > r0 and c1 > c0:
        dr = max(0, r0 - int(win.row_off))
        dc = max(0, c0 - int(win.col_off))
        patch_h = min(r1 - r0, out_h - dr)
        patch_w = min(c1 - c0, out_w - dc)
        patch = ds.read(
            1,
            window=Window(c0, r0, c1 - c0, r1 - r0),
            out_shape=(patch_h, patch_w),
        )
        arr[dr:dr + patch_h, dc:dc + patch_w] = patch
    return arr

def rasterize_ref_fraction(ref_geoms, out_shape, transform, oversample=4) -> np.ndarray:
    if len(ref_geoms) == 0:
        return np.zeros(out_shape, dtype="float32")
    if oversample <= 1:
        mask = features.rasterize(
            [(g, 1) for g in ref_geoms], out_shape=out_shape,
            transform=transform, fill=0, dtype="uint8"
        )
        return mask.astype("float32")
    h, w = out_shape
    oh, ow = h * oversample, w * oversample
    hi_transform = transform * Affine.scale(1.0 / oversample, 1.0 / oversample)
    hi = features.rasterize(
        [(g, 1) for g in ref_geoms], out_shape=(oh, ow),
        transform=hi_transform, fill=0, dtype="uint8"
    ).astype("float32")
    return hi.reshape(h, oversample, w, oversample).mean(axis=(1, 3)).astype("float32")

def aoi_mask_for_window(aoi_geom, out_shape, transform) -> np.ndarray:
    mask = features.rasterize(
        [(aoi_geom, 1)], out_shape=out_shape,
        transform=transform, fill=0, dtype="uint8"
    )
    return mask.astype(bool)

def wsf_built_pixels(arr: np.ndarray, as_of_code: int,
                     built_value_min: int = 1, nonbuilt_value: int = 0) -> np.ndarray:
    """Binary mask: pixels first detected at or before as_of_code."""
    return (arr != nonbuilt_value) & (arr >= built_value_min) & (arr <= as_of_code)

In [5]:
# ---- Load AOI tracker to enumerate cities ----
# TODO: update paths if your layout differs from the standard project structure

CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"
TRACKER_PATH = PROJECT_ROOT / "data/02_interim/aoi_tracker.csv"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Use PROJECT_ROOT (hardcoded above) as the base — not cfg["root_dir"], which
# may point to a stale drive path that no longer matches the actual mount location.
DATA_DIR = PROJECT_ROOT / cfg.get("data_dir", "data/01_raw")

# validation_configs.yaml has no top-level 'crs' key.
# Use Web Mercator (EPSG:3857): projected in metres, works globally.
# The main pipeline uses a per-city UTM zone; EPSG:3857 is an acceptable
# approximation for this sensitivity-analysis notebook.
CRS = "EPSG:3857"

TAU_FRAC   = 0.2  # ~20 m² / 100 m² for 10 m pixels; matches main pipeline default
OVERSAMPLE = int(cfg.get("raster", {}).get("preprocessing", {}).get("oversample_factor", 4))

# WSF-specific config — dataset name changed from 'wsf-tracker' (hyphen) to
# 'wsf_tracker' (underscore) in the upstream restructure.
wsf_cfg = next(
    (d for d in cfg["raster"]["datasets"] if d["name"] == "wsf_tracker"),
    None,
)
if wsf_cfg is None:
    avail = [d["name"] for d in cfg["raster"]["datasets"]]
    raise ValueError(
        f"No 'wsf_tracker' entry found under raster.datasets in validation_configs.yaml.\n"
        f"Available dataset names: {avail}"
    )

WSF_BUILT_MIN = int(wsf_cfg["binarize"].get("built_value_min", 1))
WSF_NONBUILT  = int(wsf_cfg["binarize"].get("nonbuilt_value", 0))
# Raster subdirectory name on disk; later cells fall back to the hyphen form
# if the directory was created before the upstream rename.
WSF_DIR_NAME  = wsf_cfg["name"]   # "wsf_tracker"

tracker = pd.read_csv(TRACKER_PATH, dtype=str)
tracker.columns = tracker.columns.str.strip()
tracker = tracker.apply(lambda c: c.str.strip() if c.dtype == object else c)

# Suitable-column name varies across tracker versions; match flexibly.
suitable_col = next((c for c in tracker.columns if "suitable" in c.lower()), None)
if suitable_col:
    tracker = tracker[tracker[suitable_col].str.lower() == "yes"]
else:
    print("⚠  No 'Suitable' column found in tracker — keeping all rows.")

print(f"Cities in tracker (suitable): {tracker['Dataset code'].nunique()}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"DATA_DIR exists? {DATA_DIR.exists()}")
print(f"CRS: {CRS}  |  TAU_FRAC: {TAU_FRAC}  |  OVERSAMPLE: {OVERSAMPLE}")
print(f"WSF built_value_min={WSF_BUILT_MIN}  nonbuilt_value={WSF_NONBUILT}  dir_name='{WSF_DIR_NAME}'")

Cities in tracker (suitable): 136
DATA_DIR:  /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/data/01_raw
DATA_DIR exists? True
CRS: EPSG:3857  |  TAU_FRAC: 0.2  |  OVERSAMPLE: 4
WSF built_value_min=1  nonbuilt_value=0  dir_name='wsf_tracker'


In [6]:
# ---- Per-city evaluation ----

def _find_wsf_raster(data_dir, folder: str, wsf_dir_name: str):
    """
    Locate the WSF raster for one city.

    Tries in order:
      1. Flat slug-prefixed file: raster/{slug}_{wsf_dir_name}.tif   (standard layout)
      2. Glob in raster/ for any *{wsf_dir_name}*.tif
      3. Legacy subdirectory: raster/{wsf_dir_name}/*.tif  (or hyphen form)
    Returns sorted list of candidate paths (empty if none found).
    """
    slug = folder.replace("-", "_").replace(" ", "_")
    raster_dir = data_dir / folder / "raster"

    direct = raster_dir / f"{slug}_{wsf_dir_name}.tif"
    if direct.exists():
        return [direct]

    flat = sorted(raster_dir.glob(f"*{wsf_dir_name}*.tif"))
    if flat:
        return flat

    for sub in [wsf_dir_name, wsf_dir_name.replace("_", "-")]:
        sub_dir = raster_dir / sub
        if sub_dir.exists():
            found = sorted(sub_dir.glob("*.tif"))
            if found:
                return found

    return []


def eval_wsf_for_city_all_codes(city: str, row: pd.Series, codes: list) -> list:
    """
    Run WSF validation for one city across all given as_of_code cutoffs in a
    single raster pass.

    Each city's AOI, reference, and WSF raster are opened once. Inside the tile
    loop, reference rasterization is computed once per tile; the per-cutoff
    threshold (arr <= code) is a cheap numpy comparison on the already-loaded
    array. This is ~n_cutoffs× faster than re-opening files per cutoff.

    Returns a list of metric dicts (one per code with valid coverage), or an
    empty list if any required data file is missing.
    aoi_file_name / reference_file_name may be pipe-separated lists of files.
    """
    folder   = str(row["dataset_folder_name"]).strip()
    aoi_spec = str(row.get("aoi_file_name", "")).strip()
    ref_spec = str(row.get("reference_file_name", "")).strip()

    wsf_candidates = _find_wsf_raster(DATA_DIR, folder, WSF_DIR_NAME)

    if not aoi_spec:
        warnings.warn(f"{city}: no AOI filename in tracker")
        return []
    if not ref_spec:
        warnings.warn(f"{city}: no reference filename in tracker")
        return []
    if not wsf_candidates:
        warnings.warn(f"{city}: no WSF raster found under {DATA_DIR / folder / 'raster'}")
        return []

    aoi_gdf, missing_aoi = _load_multi_file(DATA_DIR / folder / "aoi", aoi_spec, CRS)
    if aoi_gdf is None:
        warnings.warn(f"{city}: AOI file(s) not found — {', '.join(missing_aoi)}")
        return []

    ref_gdf, missing_ref = _load_multi_file(DATA_DIR / folder / "vector", ref_spec, CRS)
    if ref_gdf is None:
        warnings.warn(f"{city}: reference file(s) not found — {', '.join(missing_ref)}")
        return []

    wsf_path   = wsf_candidates[0]
    aoi_union  = aoi_gdf.geometry.union_all()
    ref_sindex = ref_gdf.sindex

    from shapely.geometry import box
    minx, miny, maxx, maxy = aoi_union.bounds
    tile_size = float(cfg.get("vector", {}).get("preprocessing", {}).get("tile_size_m", 1000))
    xs = np.arange(minx, maxx, tile_size)
    ys = np.arange(miny, maxy, tile_size)
    tiles = [
        box(x, y, x + tile_size, y + tile_size)
        for x in xs for y in ys
        if box(x, y, x + tile_size, y + tile_size).intersects(aoi_union)
    ]

    # Per-code accumulators — keyed by code
    accum = {c: {"tp": 0.0, "fp": 0.0, "fn": 0.0, "valid": 0.0} for c in codes}

    with rasterio.open(wsf_path) as src:
        ds = open_in_target_crs(src, CRS)
        nodata = ds.nodata

        for tile_geom in tiles:
            win = from_bounds(*tile_geom.bounds, transform=ds.transform)
            if win.width <= 0 or win.height <= 0:
                continue

            # Read tile once — shared across all cutoffs
            arr = _read_tile(ds, win, fill=nodata if nodata is not None else 0)
            transform = rasterio.windows.transform(win, ds.transform)
            pixel_area = _pixel_area_from_transform(transform)

            aoi_mask = aoi_mask_for_window(aoi_union, arr.shape, transform)
            valid = aoi_mask.copy()
            if nodata is not None:
                valid &= (arr != nodata)

            n_valid = int(valid.sum())
            if n_valid == 0:
                continue

            # Rasterize reference once per tile — same for all cutoffs
            possible  = list(ref_sindex.intersection(tile_geom.bounds))
            ref_tile  = ref_gdf.iloc[possible]
            ref_tile  = ref_tile[ref_tile.intersects(tile_geom)]
            f_ref     = rasterize_ref_fraction(
                list(ref_tile.geometry), arr.shape, transform, oversample=OVERSAMPLE
            )
            ref_bin = (f_ref >= TAU_FRAC)[valid]

            # Apply each cutoff threshold — cheap numpy comparison on loaded arr
            for code in codes:
                pred_bin = wsf_built_pixels(arr, code, WSF_BUILT_MIN, WSF_NONBUILT)[valid]
                a = accum[code]
                a["tp"]    += float(np.logical_and( pred_bin,  ref_bin).sum()) * pixel_area
                a["fp"]    += float(np.logical_and( pred_bin, ~ref_bin).sum()) * pixel_area
                a["fn"]    += float(np.logical_and(~pred_bin,  ref_bin).sum()) * pixel_area
                a["valid"] += n_valid * pixel_area

    results = []
    for code in codes:
        a = accum[code]
        if a["valid"] == 0:
            continue
        tp, fp, fn = a["tp"], a["fp"], a["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        results.append({
            "city":                city,
            "as_of_code":          code,
            "as_of_year":          code_to_year.get(code, None),
            "precision_area":      round(precision, 4),
            "recall_area":         round(recall, 4),
            "f1_area":             round(f1, 4),
            "tp_m2":               round(tp, 1),
            "fp_m2":               round(fp, 1),
            "fn_m2":               round(fn, 1),
            "signed_area_bias_m2": round(fp - fn, 1),
            "valid_area_m2":       round(a["valid"], 1),
        })
    return results


def eval_wsf_for_city(city: str, row: pd.Series, as_of_code: int) -> dict | None:
    """Single-cutoff wrapper used by Section 3."""
    res = eval_wsf_for_city_all_codes(city, row, [as_of_code])
    return res[0] if res else None

In [7]:
# ---- Section 1: main sweep (city-first for efficiency) ----
#
# Iterating cities in the outer loop means each city's AOI, reference, and WSF
# raster are opened exactly once. Inside the tile loop, reference rasterization
# is computed once per tile; all n_cutoffs thresholds are applied as cheap numpy
# comparisons on the already-loaded array. Compared to the cutoff-first approach
# this reduces file I/O and rasterization work by ~n_cutoffs× (~9×).

out_path = PROJECT_ROOT / "outputs/wsf_year_sensitivity.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

cities_grouped = list(tracker.groupby("Dataset code"))
n_cities  = len(cities_grouped)
results   = []

skip_permanently: set  = set()
all_issues: list = []

sweep_start = time.time()

for city_idx, (city, group) in enumerate(cities_grouped, 1):
    row = group.iloc[0]
    t0  = time.time()
    try:
        city_results = eval_wsf_for_city_all_codes(city, row, SWEEP_CODES)
        dt = time.time() - t0
        if city_results:
            results.extend(city_results)
            f1s = [r["f1_area"] for r in city_results]
            print(
                f"  [{city_idx}/{n_cities}] {city}: "
                f"{len(city_results)} cutoffs | "
                f"F1 {min(f1s):.3f}–{max(f1s):.3f}  ({dt:.0f}s)"
            )
        else:
            skip_permanently.add(city)
            all_issues.append({"city": city, "reason": "missing data"})
            print(f"  [{city_idx}/{n_cities}] {city}: SKIPPED (missing data)  ({dt:.0f}s)")
    except Exception as exc:
        dt = time.time() - t0
        skip_permanently.add(city)
        all_issues.append({"city": city, "reason": str(exc)})
        print(f"  [{city_idx}/{n_cities}] {city}: ERROR — {exc}  ({dt:.0f}s)")

    # Save after every city — robust to kernel crashes
    if results:
        pd.DataFrame(results).to_csv(out_path, index=False)

    elapsed   = time.time() - sweep_start
    remaining = n_cities - city_idx
    eta_min   = (elapsed / city_idx) * remaining / 60
    print(
        f"    elapsed {elapsed/60:.1f} min | "
        f"ETA ~{eta_min:.0f} min | "
        f"{remaining} cities left, {len(skip_permanently)} skipped"
    )

sensitivity_df = pd.DataFrame(results)
sensitivity_df.to_csv(out_path, index=False)
total_min = (time.time() - sweep_start) / 60
print(f"\nComplete in {total_min:.1f} min — saved → {out_path}")
print(f"Rows: {len(sensitivity_df)}  |  Cities: {sensitivity_df['city'].nunique()}  |  Cutoffs: {sensitivity_df['as_of_year'].nunique()}")

# ---- Issue summary ----
if all_issues:
    print(f"\n{'='*60}")
    print(f"SKIPPED / ERRORED — {len(all_issues)} cities:")
    print(f"{'='*60}")
    for issue in sorted(all_issues, key=lambda x: x["city"]):
        print(f"  {issue['city']:<30}  {issue['reason']}")
else:
    print("\n✓ All cities processed successfully.")

  [1/136] afg-ayabak: 9 cutoffs | F1 0.549–0.554  (7s)
    elapsed 0.1 min | ETA ~17 min | 135 cities left, 0 skipped
  [2/136] afg-bamyan: 9 cutoffs | F1 0.470–0.486  (5s)
    elapsed 0.2 min | ETA ~14 min | 134 cities left, 0 skipped
  [3/136] afg-bazarak: 9 cutoffs | F1 0.577–0.589  (3s)
    elapsed 0.3 min | ETA ~12 min | 133 cities left, 0 skipped
  [4/136] afg-charikar: 9 cutoffs | F1 0.642–0.652  (6s)
    elapsed 0.4 min | ETA ~12 min | 132 cities left, 0 skipped
  [5/136] afg-east-kabul: 9 cutoffs | F1 0.613–0.629  (92s)
    elapsed 1.9 min | ETA ~50 min | 131 cities left, 0 skipped
  [6/136] afg-ferozkoh: 9 cutoffs | F1 0.556–0.567  (4s)
    elapsed 2.0 min | ETA ~43 min | 130 cities left, 0 skipped
  [7/136] afg-gardez: 9 cutoffs | F1 0.506–0.520  (7s)
    elapsed 2.1 min | ETA ~39 min | 129 cities left, 0 skipped
  [8/136] afg-herat: 9 cutoffs | F1 0.668–0.685  (30s)
    elapsed 2.6 min | ETA ~42 min | 128 cities left, 0 skipped
  [9/136] afg-jalalaba: 9 cutoffs | F1 0.544–0

## Section 1 summary — Sensitivity table and flagging

In [8]:
# ---- Summary: mean F1 and mean bias per cutoff year ----

summary = (
    sensitivity_df
    .groupby(["as_of_year", "as_of_code"])
    .agg(
        n_cities             =("city",                "nunique"),
        mean_f1              =("f1_area",             "mean"),
        median_f1            =("f1_area",             "median"),
        mean_precision       =("precision_area",      "mean"),
        mean_recall          =("recall_area",         "mean"),
        mean_bias_km2        =("signed_area_bias_m2", lambda x: x.mean() / 1e6),
    )
    .reset_index()
    .sort_values("as_of_year")
)

# Flag relative to baseline (as_of_code: 19)
baseline_row = summary[summary["as_of_code"] == 19]
if len(baseline_row) == 1:
    baseline_f1 = float(baseline_row["mean_f1"].iloc[0])
    summary["f1_delta_vs_baseline"] = (summary["mean_f1"] - baseline_f1).round(4)
    summary["flagged"] = summary["f1_delta_vs_baseline"].abs() > 0.05
    print(f"Baseline (as_of_code=19) mean F1: {baseline_f1:.4f}")
else:
    summary["f1_delta_vs_baseline"] = np.nan
    summary["flagged"] = False
    print("Note: as_of_code=19 not in sweep (year 2019 may not be in data range).")

print("\n=== WSF Year-Code Sensitivity Summary ===")
display(
    summary[[
        "as_of_year", "as_of_code", "n_cities",
        "mean_f1", "median_f1", "mean_precision", "mean_recall",
        "mean_bias_km2", "f1_delta_vs_baseline", "flagged"
    ]].round(4)
)

flagged = summary[summary["flagged"]]
if len(flagged):
    print(f"\n⚠  {len(flagged)} cutoff(s) differ from baseline by >0.05 F1:")
    for _, r in flagged.iterrows():
        print(f"   {int(r.as_of_year)} (code={int(r.as_of_code)}): "
              f"mean F1={r.mean_f1:.4f}  delta={r.f1_delta_vs_baseline:+.4f}")
else:
    print("\n✓  No cutoff differs from baseline by more than 0.05 F1.")

Baseline (as_of_code=19) mean F1: 0.5973

=== WSF Year-Code Sensitivity Summary ===


,as_of_year,as_of_code,n_cities,mean_f1,median_f1,mean_precision,mean_recall,mean_bias_km2,f1_delta_vs_baseline,flagged
0,2019.0,6,134,0.6105,0.6293,0.4732,0.9103,4.8896,0.0132,False
1,2020.0,8,134,0.6170,0.6324,0.4683,0.9473,5.1311,0.0197,False
2,2021.0,10,134,0.6141,0.6315,0.4608,0.9663,5.3444,0.0168,False
3,2022.0,12,134,0.6082,0.6228,0.4575,0.9741,5.5468,0.0109,False
4,2023.0,14,134,0.6062,0.6216,0.4503,0.9845,5.6927,0.0090,False
5,2024.0,16,134,0.6027,0.6188,0.4460,0.9888,5.8042,0.0055,False
6,2025.0,18,134,0.5995,0.6159,0.4425,0.9912,5.9085,0.0022,False
7,2025.5,19,134,0.5973,0.6138,0.4403,0.9925,5.9926,0.0000,False
8,2026.0,20,134,0.5973,0.6138,0.4403,0.9925,5.9926,0.0000,False



✓  No cutoff differs from baseline by more than 0.05 F1.


In [9]:
# ---- Interpretation: is as_of_code=19 capturing all years or cutting off early? ----

max_code_in_data = max(code_to_year.keys())
max_year_in_data = code_to_year[max_code_in_data]

print("=" * 55)
print("as_of_code=19 interpretation")
print("=" * 55)

if 19 > max_code_in_data:
    print(f"as_of_code=19 EXCEEDS the maximum code ({max_code_in_data} = {max_year_in_data}).")
    print("→ Current config captures ALL years — no temporal bias introduced.")
elif 19 in code_to_year:
    mapped_year = code_to_year[19]
    # int() handles float years (e.g. 2024.5) when constructing the year range.
    next_full_year = int(mapped_year) + 1
    max_full_year  = int(max_year_in_data)
    years_missing  = list(range(next_full_year, max_full_year + 1))
    print(f"as_of_code=19 → {mapped_year} in the bi-annual encoding.")
    if years_missing:
        print(f"→ Full calendar years excluded from built-up mask: {years_missing}")
        print("→ Buildings added in those years appear as False Negatives.")
        print("→ Consider updating as_of_code in configs/validation_configs.yaml.")
    else:
        print("→ No full calendar years are excluded — gap to max code is < 1 year.")
        print("→ Temporal bias from the current as_of_code is negligible.")
else:
    print(f"as_of_code=19 is not in the confirmed mapping.")
    print(f"Valid codes: {sorted(code_to_year.keys())}")

as_of_code=19 interpretation
as_of_code=19 → 2025.5 in the bi-annual encoding.
→ Full calendar years excluded from built-up mask: [2026]
→ Buildings added in those years appear as False Negatives.
→ Consider updating as_of_code in configs/validation_configs.yaml.


---

## Section 2: Urban Growth Rate vs. Tile F1 Accuracy

**Purpose:** Test whether cities with faster urban growth (more WSF pixels added between the reference image year and Jan 2025) have systematically lower F1 scores, indicating that temporal mismatch — not just spatial accuracy — is a meaningful driver of validation error.

**Inputs:**
- `Reference dataset overview.xlsx` — maps each city to the year of its reference imagery
- WSF Tracker rasters on disk — used to compute built-up pixel counts at two time points
- `vector_all_cities_merged.xlsx` — per-city F1 scores from the vector validation pipeline

**Method:**
1. For each city, derive `wsf_code_early` from the reference image year using `code = round((ref_year - 2015) × 2)`, clipped to [1, 20].
2. Count built-up WSF pixels at `wsf_code_early` and `wsf_code_late = 19` (Jan 2025).
3. Merge with vector F1. Correlate growth rate with F1 across datasets, split by SpaceNet7 vs. other.

### Cell 1 — Load reference image years

In [19]:
import warnings
import io
import re
from google.colab import files as _colab_files

# Upload the Excel export of "Reference dataset overview".
# In Google Sheets: File → Download → Microsoft Excel (.xlsx), then upload here.
print("Upload the Excel export of 'Reference dataset overview':")
_uploaded = _colab_files.upload()
_fname    = next(iter(_uploaded))
ref_raw   = pd.read_excel(
    io.BytesIO(_uploaded[_fname]),
    sheet_name="Reference Dataset overview",
    dtype=str,
)
ref_raw.columns = ref_raw.columns.str.strip()

# Column detection — prefer "year (of the image)" over any other year column
code_col = next(c for c in ref_raw.columns if "dataset" in c.lower() and "code" in c.lower())
year_col = next(
    (c for c in ref_raw.columns if "year" in c.lower() and "image" in c.lower()),
    next((c for c in ref_raw.columns if "year" in c.lower()), None),
)
if year_col is None:
    raise ValueError(f"No year column found. Columns: {list(ref_raw.columns)}")
print(f"Using columns: code='{code_col}', year='{year_col}'")

_TIMESTAMP_RE = re.compile(r"^(\d{4})-\d{2}-\d{2}")

# Parse year for every row — datetime values are image acquisition dates,
# so we extract just the year. Cities with multiple rows get all their years
# collected, then reduced to one below.
city_years: dict[str, list[int]] = {}   # city -> [year, year, ...]
skipped_unclear: list[str] = []
skipped_missing: list[str] = []

for _, row in ref_raw.iterrows():
    city     = str(row[code_col]).strip()
    raw_year = str(row[year_col]).strip()

    if not city or city.lower() in {"nan", ""}:
        continue

    if raw_year.lower() in {"various", "nan", "", "n/a", "tbc"}:
        skipped_missing.append(city)
        continue

    if raw_year.lower() in {"unclear"}:
        skipped_unclear.append(city)
        continue

    # Full datetime stored as image date — extract the year
    m = _TIMESTAMP_RE.match(raw_year)
    if m:
        city_years.setdefault(city, []).append(int(m.group(1)))
        continue

    # Plain four-digit year
    try:
        city_years.setdefault(city, []).append(int(float(raw_year)))
    except ValueError:
        skipped_unclear.append(f"{city} ('{raw_year}')")

# For cities with multiple rows, use the most recent year.
# The latest acquisition is typically the one used for validation.
ref_year_map: dict[str, int] = {c: max(yrs) for c, yrs in city_years.items()}

multi_year = {c: sorted(set(yrs)) for c, yrs in city_years.items() if len(set(yrs)) > 1}
print(f"Cities parsed from sheet:             {len(ref_year_map)}")
if multi_year:
    print(f"Cities with multiple acquisition years (using most recent):")
    for c, yrs in sorted(multi_year.items()):
        print(f"   {c}: {yrs} → {max(yrs)}")

# Cross-check: keep only cities in WSF sweep results.
_wsf_csv = PROJECT_ROOT / "outputs/wsf_year_sensitivity.csv"
if "sensitivity_df" in dir() and not sensitivity_df.empty:
    wsf_cities = set(sensitivity_df["city"].unique())
    wsf_source = "sensitivity_df (in memory)"
elif _wsf_csv.exists():
    wsf_cities = set(pd.read_csv(_wsf_csv)["city"].unique())
    wsf_source = str(_wsf_csv)
else:
    wsf_cities = set()
    wsf_source = "none — run Section 1 first"

if wsf_cities:
    not_in_wsf   = sorted(k for k in ref_year_map if k not in wsf_cities)
    in_wsf_no_year = sorted(c for c in wsf_cities if c not in ref_year_map)
    ref_year_map = {k: v for k, v in ref_year_map.items() if k in wsf_cities}
    print(f"\nAfter cross-check with WSF results ({wsf_source}):")
    print(f"  Matched: {len(ref_year_map)} cities")
    print(f"  In sheet but not in WSF sweep: {len(not_in_wsf)}")
    if in_wsf_no_year:
        print(f"  In WSF sweep but no image year in sheet ({len(in_wsf_no_year)}):")
        for c in in_wsf_no_year:
            print(f"    {c}")
else:
    print(f"\n⚠ No WSF sweep results found ({wsf_source}) — keeping all {len(ref_year_map)} cities.")

if skipped_unclear:
    print(f"\n⚠ {len(set(skipped_unclear))} cities skipped (year='unclear' in sheet):")
    print(f"   {', '.join(sorted(set(skipped_unclear)))}")
if skipped_missing:
    print(f"\n⚠ {len(set(skipped_missing))} cities skipped (year blank/various/n/a):")
    print(f"   {', '.join(sorted(set(skipped_missing)))}")

print(f"\nFinal ref_year_map: {len(ref_year_map)} cities")
print("\nSample (first 10):")
for city, yr in list(ref_year_map.items())[:10]:
    print(f"  {city}: {yr}")

Upload the Excel export of 'Reference dataset overview':


Saving Reference dataset overview.xlsx to Reference dataset overview (2).xlsx
Using columns: code='Dataset code', year='Year (of the image)'
Cities parsed from sheet:             281
Cities with multiple acquisition years (using most recent):
   bgd-rohingya: [2018, 2020] → 2020
   col-medellin: [2024, 2025] → 2025
   gha-accra: [2019, 2020, 2024] → 2024
   gha-bole: [2021, 2022] → 2022
   gha-damongo: [2020, 2022] → 2022
   hti-canaan: [2020, 2021] → 2021
   jpn-hiroshima: [2018, 2019] → 2019
   ken-kakuma: [2022, 2023, 2025] → 2025
   ken-nairobi: [2022, 2023, 2024, 2025] → 2025
   ken-onwordhei: [2024, 2025] → 2025
   lbr-monrovia: [2020, 2021, 2024] → 2024
   mex-acapulco: [2023, 2024] → 2024
   mwi-mangochi: [2020, 2021] → 2021
   phl-bagamanoc: [2021, 2023] → 2023
   phl-barangay: [2017, 2018, 2019] → 2019
   phl-catanduanes: [2021, 2022, 2025] → 2025
   sdn-blue-nile: [2018, 2019, 2020] → 2020
   sle-freetown: [2024, 2025, 2026] → 2026
   sxm-sint-maarten: [2017, 2020] → 2020
  

### Cell 2 — WSF growth rate per city

In [20]:
WSF_CODE_LATE = 19   # mid-2025 (current as_of_code in configs/validation_configs.yaml)
GROWTH_CAP    = 5.0  # 500 % cap to handle near-zero denominators

# bi-annual encoding: code = round((year - 2016) * 2), range [1, 20]
# code 1 = mid-2016, code 2 = end-2016, code 3 = mid-2017, ...
def ref_year_to_wsf_code(year: int) -> int:
    return int(np.clip(round((year - 2016) * 2), 1, 20))

def count_wsf_built_pixels(wsf_path, as_of_code: int,
                            built_value_min: int = 1,
                            nonbuilt_value: int = 0) -> int:
    """Read full raster and count pixels settled at or before as_of_code."""
    with rasterio.open(wsf_path) as src:
        arr = src.read(1)
        nodata = src.nodata
        valid = np.ones(arr.shape, dtype=bool)
        if nodata is not None:
            valid &= (arr != nodata)
        mask = wsf_built_pixels(arr, as_of_code, built_value_min, nonbuilt_value)
        return int((mask & valid).sum())

growth_rows = []

for city, ref_year in ref_year_map.items():
    # Look up dataset folder from tracker
    city_rows = tracker[tracker["Dataset code"] == city]
    if city_rows.empty:
        warnings.warn(f"{city}: not found in AOI tracker — skipping.")
        continue

    folder = str(city_rows.iloc[0]["dataset_folder_name"]).strip()

    # Use the same helper as the sensitivity sweep for consistent path resolution.
    wsf_candidates = _find_wsf_raster(DATA_DIR, folder, WSF_DIR_NAME)
    if not wsf_candidates:
        warnings.warn(f"{city}: no WSF raster found under {DATA_DIR / folder / 'raster'} — skipping.")
        continue

    wsf_path = wsf_candidates[0]
    wsf_code_early = ref_year_to_wsf_code(ref_year)

    try:
        pixels_early = count_wsf_built_pixels(
            wsf_path, wsf_code_early, WSF_BUILT_MIN, WSF_NONBUILT
        )
        pixels_late = count_wsf_built_pixels(
            wsf_path, WSF_CODE_LATE, WSF_BUILT_MIN, WSF_NONBUILT
        )
    except Exception as exc:
        warnings.warn(f"{city}: raster read failed — {exc}")
        continue

    if pixels_early == 0:
        warnings.warn(f"{city}: zero built pixels at code {wsf_code_early} — "
                      f"growth rate undefined; capping at {GROWTH_CAP:.0%}.")
        growth_rate = GROWTH_CAP
    else:
        growth_rate = min((pixels_late - pixels_early) / pixels_early, GROWTH_CAP)

    growth_rows.append({
        "city":               city,
        "ref_image_year":     ref_year,
        "wsf_code_early":     wsf_code_early,
        "wsf_code_late":      WSF_CODE_LATE,
        "temporal_gap_years": (WSF_CODE_LATE - wsf_code_early) / 2.0,
        "pixels_early":       pixels_early,
        "pixels_late":        pixels_late,
        "growth_rate":        round(growth_rate, 4),
    })

growth_df = pd.DataFrame(growth_rows)
print(f"Growth rates computed for {len(growth_df)} cities.")
print(f"Capped at {GROWTH_CAP:.0%}: "
      f"{(growth_df['growth_rate'] == GROWTH_CAP).sum()} city/cities.")

Growth rates computed for 64 cities.
Capped at 500%: 0 city/cities.


### Cell 3 — Year selection log

> ⚠ **Review this table before interpreting results.**
> Verify that `ref_image_year` matches what you expect from the reference data documentation, and that `wsf_code_early` is the correct bi-annual code for that year. Cities where `temporal_gap_years` is 0 or negative should be investigated — they may have a reference image year that is more recent than the WSF cutoff.

In [21]:
log_cols = ["city", "ref_image_year", "wsf_code_early", "wsf_code_late", "temporal_gap_years"]
log_df = growth_df[log_cols].sort_values("city").reset_index(drop=True)

print(f"{'city':<25} {'ref_year':>8} {'code_early':>10} {'code_late':>9} {'gap_years':>9}")
print("-" * 65)
for _, r in log_df.iterrows():
    flag = "  ← ⚠" if r.temporal_gap_years <= 0 else ""
    print(
        f"{r.city:<25} {int(r.ref_image_year):>8} {int(r.wsf_code_early):>10} "
        f"{int(r.wsf_code_late):>9} {r.temporal_gap_years:>9.1f}{flag}"
    )

suspicious = log_df[log_df["temporal_gap_years"] <= 0]
if not suspicious.empty:
    print(f"\n⚠ {len(suspicious)} city/cities have zero or negative temporal gap — investigate:")
    print(suspicious.to_string(index=False))

city                      ref_year code_early code_late gap_years
-----------------------------------------------------------------
bgd-rohingya                  2020          8        19       5.5
blz-burrell-boom              2020          8        19       5.5
bra-nova-sussuarana           2022         12        19       3.5
col-san-antonio-de-prado      2024         16        19       1.5
gha-accra                     2024         16        19       1.5
gha-aiyim-sraha               2021         10        19       4.5
gha-dansoman                  2021         10        19       4.5
gha-nawuni                    2022         12        19       3.5
gha-sawla-tuna                2024         16        19       1.5
gha-wa                        2024         16        19       1.5
jam-kingston                  2021         10        19       4.5
jam-saint-catherine           2021         10        19       4.5
jpn-ashiya-hama               2019          6        19       6.5
jpn-hirosh

### Cell 4 — OBT temporal encoding

**Finding from pipeline audit (`src/download/raster.py`, `src/validate/raster_runner.py`, `configs/validation_configs.yaml`):**

OBT year selection is **hardcoded to 2023** in the validation config:

```yaml
# configs/validation_configs.yaml
- name: obt
  year: 2023          # ← fixed; raster_runner globs {slug}_obt_2023*.tif
```

**Download side** (`OBTRunner.run()`): loops over `GoogleOBTConfig.years = [2016, …, 2023]` and saves one GeoTIFF per year as `{slug}_obt_{year}.tif`. All annual files are on disk.

**Validation side** (`raster_runner.py` L114–122): uses the `year` field to build the glob pattern `{slug}_obt_2023*` and picks the first match. It is **not** matched dynamically to the reference image year.

**Implication for growth rate analysis:** Computing OBT growth rate requires reading two annual files per city — `{slug}_obt_{ref_year}.tif` (early) and `{slug}_obt_2023.tif` (late). Both files exist on disk for any reference year in [2016, 2023]. This is a straightforward two-file comparison, but it requires explicit year-pair logic not yet in the pipeline.

In [ ]:
# TODO: implement OBT growth rate comparison.
#
# For each city with ref_image_year in [2016, 2023]:
#   early_path = DATA_DIR / folder / "raster" / f"{slug}_obt_{ref_year}.tif"
#   late_path  = DATA_DIR / folder / "raster" / f"{slug}_obt_2023.tif"
#   Compare fractional building coverage at both years (band 1 = building_presence).
#   growth_rate = (mean_presence_late - mean_presence_early) / mean_presence_early
#
# This requires two-file explicit extraction — not yet implemented in the pipeline.
# The annual files already exist on disk (downloaded by OBTRunner).
print("OBT growth rate: TODO — see markdown above for implementation notes.")

### Cell 5 — Merge WSF growth rates with vector F1

In [26]:
import warnings
import io
import re
from google.colab import files as _colab_files

# Upload the Excel export of "Reference dataset overview".
# In Google Sheets: File → Download → Microsoft Excel (.xlsx), then upload here.
print("Upload the Excel export of 'Reference dataset overview':")
_uploaded = _colab_files.upload()
_fname    = next(iter(_uploaded))
ref_raw   = pd.read_excel(
    io.BytesIO(_uploaded[_fname]),
    sheet_name="Reference Dataset overview",
    dtype=str,
)
ref_raw.columns = ref_raw.columns.str.strip()

# Column detection — prefer "year (of the image)" over any other year column
code_col = next(c for c in ref_raw.columns if "dataset" in c.lower() and "code" in c.lower())
year_col = next(
    (c for c in ref_raw.columns if "year" in c.lower() and "image" in c.lower()),
    next((c for c in ref_raw.columns if "year" in c.lower()), None),
)
if year_col is None:
    raise ValueError(f"No year column found. Columns: {list(ref_raw.columns)}")
print(f"Using columns: code='{code_col}', year='{year_col}'")

_TIMESTAMP_RE = re.compile(r"^(\d{4})-\d{2}-\d{2}")

city_years:      dict[str, list[int]] = {}
cities_in_sheet: set[str] = set()   # every city seen in the sheet, regardless of year quality
skipped_unclear: list[str] = []
skipped_missing: list[str] = []

for _, row in ref_raw.iterrows():
    city     = str(row[code_col]).strip()
    raw_year = str(row[year_col]).strip()

    if not city or city.lower() in {"nan", ""}:
        continue

    cities_in_sheet.add(city)   # record presence regardless of year value

    if raw_year.lower() in {"various", "nan", "", "n/a", "tbc"}:
        skipped_missing.append(city)
        continue
    if raw_year.lower() == "unclear":
        skipped_unclear.append(city)
        continue

    m = _TIMESTAMP_RE.match(raw_year)
    if m:
        city_years.setdefault(city, []).append(int(m.group(1)))
        continue

    try:
        city_years.setdefault(city, []).append(int(float(raw_year)))
    except ValueError:
        skipped_unclear.append(f"{city} ('{raw_year}')")

# For cities with multiple rows, use the most recent year.
ref_year_map: dict[str, int] = {c: max(yrs) for c, yrs in city_years.items()}

multi_year = {c: sorted(set(yrs)) for c, yrs in city_years.items() if len(set(yrs)) > 1}
print(f"\nCities parsed from sheet: {len(ref_year_map)}")
if multi_year:
    print(f"Cities with multiple acquisition years (using most recent):")
    for c, yrs in sorted(multi_year.items()):
        print(f"   {c}: {yrs} → {max(yrs)}")

# ---- Cross-check against WSF sweep results ----
_wsf_csv = PROJECT_ROOT / "outputs/wsf_year_sensitivity.csv"
if "sensitivity_df" in dir() and not sensitivity_df.empty:
    wsf_cities = set(sensitivity_df["city"].unique())
    wsf_source = "sensitivity_df (in memory)"
elif _wsf_csv.exists():
    wsf_cities = set(pd.read_csv(_wsf_csv)["city"].unique())
    wsf_source = str(_wsf_csv)
else:
    wsf_cities = set()
    wsf_source = "none — run Section 1 first"

SPACENET7_ASSUMED_YEAR = 2020

if wsf_cities:
    not_in_wsf     = sorted(k for k in ref_year_map if k not in wsf_cities)
    ref_year_map   = {k: v for k, v in ref_year_map.items() if k in wsf_cities}
    in_wsf_no_year = sorted(c for c in wsf_cities if c not in ref_year_map)

    # Split cities missing a year into two groups:
    #   - not in sheet at all → assume SpaceNet7 (year = 2020)
    #   - in sheet but year was "unclear" or blank → genuinely unknown, exclude
    assumed_sn7    = [c for c in in_wsf_no_year if c not in cities_in_sheet]
    year_unknown   = [c for c in in_wsf_no_year if c in cities_in_sheet]

    for c in assumed_sn7:
        ref_year_map[c] = SPACENET7_ASSUMED_YEAR

    print(f"\nAfter cross-check with WSF results ({wsf_source}):")
    print(f"  Confirmed year from sheet:          {len(ref_year_map) - len(assumed_sn7)}")
    print(f"  Assumed SpaceNet7 (year={SPACENET7_ASSUMED_YEAR}):      {len(assumed_sn7)}")
    print(f"  In WSF sweep, year unclear/missing: {len(year_unknown)}  (excluded)")
    print(f"  In sheet but not in WSF sweep:      {len(not_in_wsf)}  (excluded)")
    print(f"  Total in ref_year_map:              {len(ref_year_map)}")

    if assumed_sn7:
        print(f"\n⚠ ASSUMPTION: {len(assumed_sn7)} cities absent from reference overview "
              f"→ assumed SpaceNet7, ref_image_year={SPACENET7_ASSUMED_YEAR}.")
        for c in assumed_sn7:
            print(f"    {c}")

    if year_unknown:
        print(f"\n⚠ {len(year_unknown)} cities in WSF sweep have 'unclear' or missing year "
              f"in the reference overview — excluded from Sections 2 & 3:")
        for c in sorted(year_unknown):
            print(f"    {c}")
else:
    print(f"\n⚠ No WSF sweep results found ({wsf_source}) — keeping all {len(ref_year_map)} cities.")

if skipped_unclear:
    print(f"\n⚠ {len(set(skipped_unclear))} sheet rows skipped (year='unclear').")
if skipped_missing:
    print(f"\n⚠ {len(set(skipped_missing))} sheet rows skipped (year blank/various/n/a).")

print(f"\nFinal ref_year_map: {len(ref_year_map)} cities")
print("\nSample (first 10):")
for city, yr in list(ref_year_map.items())[:10]:
    print(f"  {city}: {yr}")

Upload the Excel export of 'Reference dataset overview':


Saving Reference dataset overview.xlsx to Reference dataset overview (4).xlsx
Using columns: code='Dataset code', year='Year (of the image)'

Cities parsed from sheet: 281
Cities with multiple acquisition years (using most recent):
   bgd-rohingya: [2018, 2020] → 2020
   col-medellin: [2024, 2025] → 2025
   gha-accra: [2019, 2020, 2024] → 2024
   gha-bole: [2021, 2022] → 2022
   gha-damongo: [2020, 2022] → 2022
   hti-canaan: [2020, 2021] → 2021
   jpn-hiroshima: [2018, 2019] → 2019
   ken-kakuma: [2022, 2023, 2025] → 2025
   ken-nairobi: [2022, 2023, 2024, 2025] → 2025
   ken-onwordhei: [2024, 2025] → 2025
   lbr-monrovia: [2020, 2021, 2024] → 2024
   mex-acapulco: [2023, 2024] → 2024
   mwi-mangochi: [2020, 2021] → 2021
   phl-bagamanoc: [2021, 2023] → 2023
   phl-barangay: [2017, 2018, 2019] → 2019
   phl-catanduanes: [2021, 2022, 2025] → 2025
   sdn-blue-nile: [2018, 2019, 2020] → 2020
   sle-freetown: [2024, 2025, 2026] → 2026
   sxm-sint-maarten: [2017, 2020] → 2020
   uga-kampal

In [39]:
import io
from google.colab import files as _colab_files

# Upload the vector validation results Excel file.
print("Upload 'vector_all_cities_merged.xlsx':")
_uploaded = _colab_files.upload()
_fname    = next(iter(_uploaded))
_buf      = io.BytesIO(_uploaded[_fname])

f1_raw = pd.read_excel(_buf, sheet_name="vector_all_cities_merged", dtype=str)
f1_raw.columns = f1_raw.columns.str.strip()

# Normalise city column name
city_col_f1 = next(
    (c for c in f1_raw.columns if "city" in c.lower() or "dataset" in c.lower()),
    f1_raw.columns[0],
)
f1_raw = f1_raw.rename(columns={city_col_f1: "city"})

# Identify F1 and dataset columns
dataset_col = next(c for c in f1_raw.columns if "dataset" in c.lower() and c != "city")
f1_col      = next(c for c in f1_raw.columns if "f1" in c.lower())

f1_df = f1_raw[["city", dataset_col, f1_col]].copy()
f1_df.columns = ["city", "dataset", "f1"]
f1_df["f1"] = pd.to_numeric(f1_df["f1"], errors="coerce")
f1_df = f1_df.dropna(subset=["f1"])

# ---- SpaceNet7 flag ----
# "new regions" sheet: "city" = non-SpaceNet7, "new city" = SpaceNet7.
# Require "new" + "city" to avoid matching the plain "city" column first.
_buf.seek(0)
sn7_raw = pd.read_excel(_buf, sheet_name="new regions", dtype=str)
sn7_raw.columns = sn7_raw.columns.str.strip()
print(f"Columns in 'new regions' sheet: {list(sn7_raw.columns)}")

_new_city_col = next(
    (c for c in sn7_raw.columns if c.strip().lower() == "new city"),
    next(
        (c for c in sn7_raw.columns if "new" in c.lower() and "city" in c.lower()),
        None,
    ),
)
if _new_city_col is None:
    raise ValueError(
        f"Cannot find 'new city' column in 'new regions' sheet.\n"
        f"Available columns: {list(sn7_raw.columns)}"
    )
print(f"SpaceNet7 column: '{_new_city_col}'")

sn7_cities = set(sn7_raw[_new_city_col].dropna().str.strip().unique())
f1_df["is_spacenet7"] = f1_df["city"].isin(sn7_cities)
print(f"SpaceNet7 cities in 'new city' column: {len(sn7_cities)}")

# ---- Reconcile: compute growth rates for SpaceNet7 cities missing from growth_df ----
# These cities are absent because either:
#   (a) Cell 2 was not re-run after Cell 1 was updated, or
#   (b) they have "unclear" year in the reference overview, so Cell 1 put them in
#       year_unknown (excluded) rather than assumed_sn7 (year=2020).
# Either way, SpaceNet7 cities should use ref_year=2020 — compute them here.
sn7_in_f1     = set(f1_df[f1_df["is_spacenet7"]]["city"].unique())
sn7_in_growth = set(growth_df["city"].unique()) & sn7_cities
sn7_missing   = sn7_in_f1 - sn7_in_growth

print(f"\nSpaceNet7 cities in f1_df:    {len(sn7_in_f1)}")
print(f"SpaceNet7 cities in growth_df: {len(sn7_in_growth)}")
print(f"SpaceNet7 cities to compute:   {len(sn7_missing)}")

if sn7_missing:
    print(f"\nComputing WSF growth rates for {len(sn7_missing)} SpaceNet7 cities (ref_year={SPACENET7_ASSUMED_YEAR})...")
    extra_rows = []
    sn7_skipped = []

    for city in sorted(sn7_missing):
        city_rows = tracker[tracker["Dataset code"] == city]
        if city_rows.empty:
            sn7_skipped.append(f"{city}: not in AOI tracker")
            continue

        folder = str(city_rows.iloc[0]["dataset_folder_name"]).strip()
        wsf_candidates = _find_wsf_raster(DATA_DIR, folder, WSF_DIR_NAME)
        if not wsf_candidates:
            sn7_skipped.append(f"{city}: no WSF raster under {DATA_DIR / folder / 'raster'}")
            continue

        wsf_path       = wsf_candidates[0]
        wsf_code_early = ref_year_to_wsf_code(SPACENET7_ASSUMED_YEAR)  # 2020 → code 8

        try:
            pixels_early = count_wsf_built_pixels(wsf_path, wsf_code_early, WSF_BUILT_MIN, WSF_NONBUILT)
            pixels_late  = count_wsf_built_pixels(wsf_path, WSF_CODE_LATE,  WSF_BUILT_MIN, WSF_NONBUILT)
        except Exception as exc:
            sn7_skipped.append(f"{city}: raster read failed — {exc}")
            continue

        if pixels_early == 0:
            growth_rate = GROWTH_CAP
        else:
            growth_rate = min((pixels_late - pixels_early) / pixels_early, GROWTH_CAP)

        extra_rows.append({
            "city":               city,
            "ref_image_year":     SPACENET7_ASSUMED_YEAR,
            "wsf_code_early":     wsf_code_early,
            "wsf_code_late":      WSF_CODE_LATE,
            "temporal_gap_years": (WSF_CODE_LATE - wsf_code_early) / 2.0,
            "pixels_early":       pixels_early,
            "pixels_late":        pixels_late,
            "growth_rate":        round(growth_rate, 4),
        })
        print(f"  {city}: growth_rate={growth_rate:.4f}")

    if extra_rows:
        growth_df = pd.concat([growth_df, pd.DataFrame(extra_rows)], ignore_index=True)
        print(f"\nAdded {len(extra_rows)} SpaceNet7 rows to growth_df.")
    if sn7_skipped:
        print(f"\n⚠ {len(sn7_skipped)} SpaceNet7 cities skipped:")
        for s in sn7_skipped:
            print(f"  {s}")

# ---- Merge with growth rates ----
merged_df = f1_df.merge(
    growth_df[["city", "ref_image_year", "wsf_code_early", "temporal_gap_years", "growth_rate"]],
    on="city",
    how="inner",
)

print(f"\nDatasets in merged:      {sorted(merged_df['dataset'].unique())}")
print(f"F1 rows loaded:          {len(f1_df)}")
print(f"After merge with growth: {len(merged_df)}")
print(f"Unique cities in merged: {merged_df['city'].nunique()}")
print(f"  of which SpaceNet7:    {merged_df[merged_df['is_spacenet7']]['city'].nunique()}")
print(f"  of which non-SN7:      {merged_df[~merged_df['is_spacenet7']]['city'].nunique()}")
display(merged_df.head(50))

Upload 'vector_all_cities_merged.xlsx':


Saving vector_all_cities_merged.xlsx to vector_all_cities_merged (7).xlsx
Columns in 'new regions' sheet: ['city', 'city.1', 'New city']
SpaceNet7 column: 'New city'
SpaceNet7 cities in 'new city' column: 49

SpaceNet7 cities in f1_df:    49
SpaceNet7 cities in growth_df: 0
SpaceNet7 cities to compute:   49

Computing WSF growth rates for 49 SpaceNet7 cities (ref_year=2020)...
  alg-tindouf: growth_rate=0.3382
  are-abudhabi: growth_rate=1.0286
  aus-melbourne: growth_rate=1.5657
  bgd-dhaka: growth_rate=0.1137
  bra-manaus: growth_rate=0.0527
  bra-saopaulo: growth_rate=0.0875
  chl-calama: growth_rate=0.1133
  chn-chengdu: growth_rate=0.1623
  chn-guangzhou: growth_rate=0.4590
  chn-lujiang: growth_rate=0.3036
  chn-shanghai: growth_rate=0.0617
  chn-wuhan: growth_rate=0.1974
  chn-yangzhou: growth_rate=0.0798
  chn-zhuhai: growth_rate=0.1558
  egy-cairo: growth_rate=0.2266
  gbr-birmingham: growth_rate=0.0405
  gbr-london: growth_rate=0.1956
  gha-kumasi: growth_rate=0.3501
  ind-mu

,city,dataset,f1,is_spacenet7,ref_image_year,wsf_code_early,temporal_gap_years,growth_rate
0,alg-tindouf,gba,0.1795,True,2020,8,5.5,0.3382
1,alg-tindouf,globfp,0.1787,True,2020,8,5.5,0.3382
2,alg-tindouf,overture,0.1726,True,2020,8,5.5,0.3382
3,are-abudhabi,gba,0.4223,True,2020,8,5.5,1.0286
4,are-abudhabi,globfp,0.4221,True,2020,8,5.5,1.0286
5,are-abudhabi,overture,0.4236,True,2020,8,5.5,1.0286
6,aus-melbourne,gba,0.2131,True,2020,8,5.5,1.5657
7,aus-melbourne,globfp,0.1223,True,2020,8,5.5,1.5657
8,aus-melbourne,overture,0.2145,True,2020,8,5.5,1.5657
9,bgd-dhaka,gba,0.0877,True,2020,8,5.5,0.1137


### Cell 6 — Scatter plot: growth rate vs. F1, by dataset and group

In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import numpy as np

DATASET_COLORS = {
    "GBA":      "#1f77b4",
    "GlobFP":   "#ff7f0e",
    "Overture": "#2ca02c",
}
_colors_lower = {k.lower(): (k, v) for k, v in DATASET_COLORS.items()}

merged_df["dataset_label"] = merged_df["dataset"].str.strip()

_unknown_ds = {d for d in merged_df["dataset_label"].unique() if d.lower() not in _colors_lower}
if _unknown_ds:
    print(f"⚠ Datasets not in DATASET_COLORS (skipped): {sorted(_unknown_ds)}")

_late_year_label = f"mid-{int(code_to_year.get(WSF_CODE_LATE, 2025))}"

JITTER_AMOUNT = 0.10   # horizontal jitter in years — only used for temporal gap figure


def _make_fig(x_col, x_label, title, x_start=0, add_jitter=False,
              panels=None):
    """
    Interactive plotly figure with one panel per entry in `panels`.
    panels: list of (is_spacenet7: bool, panel_title: str).
    Defaults to both SpaceNet7 and non-SpaceNet7 side by side.
    """
    if panels is None:
        panels = [(True, "SpaceNet7 cities"), (False, "Non-SpaceNet7 cities")]

    n_cols = len(panels)
    fig = make_subplots(
        rows=1, cols=n_cols,
        subplot_titles=[title for _, title in panels],
        shared_yaxes=(n_cols > 1),
        horizontal_spacing=0.10,
    )

    shown_legend: set = set()

    for col_idx, (is_sn7, _panel_title) in enumerate(panels, 1):
        subset = merged_df[merged_df["is_spacenet7"] == is_sn7]

        for ds_key_lower, (ds_display, ds_color) in _colors_lower.items():
            ds_data = subset[
                subset["dataset_label"].str.lower() == ds_key_lower
            ].dropna(subset=[x_col, "f1"])
            if ds_data.empty:
                continue

            x_true = ds_data[x_col].values.astype(float)
            y_vals = ds_data["f1"].values.astype(float)

            if add_jitter:
                seed   = abs(hash(ds_key_lower)) % 10_000 + col_idx * 31
                rng    = np.random.default_rng(seed)
                x_plot = x_true + rng.normal(0, JITTER_AMOUNT, len(x_true))
            else:
                x_plot = x_true

            show_legend = ds_display not in shown_legend
            shown_legend.add(ds_display)

            fig.add_trace(
                go.Scatter(
                    x=x_plot, y=y_vals,
                    mode="markers",
                    name=ds_display,
                    legendgroup=ds_display,
                    showlegend=show_legend,
                    marker=dict(
                        color=ds_color, size=9, opacity=0.75,
                        line=dict(width=0.5, color="white"),
                    ),
                    customdata=np.column_stack([
                        ds_data["city"].values,
                        ds_data["ref_image_year"].values,
                        x_true,
                    ]),
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        f"{x_label}: %{{customdata[2]:.3f}}<br>"
                        "F1 score: %{y:.3f}<br>"
                        "Ref year: %{customdata[1]:.0f}"
                        "<extra></extra>"
                    ),
                ),
                row=1, col=col_idx,
            )

            if len(ds_data) >= 3:
                slope, intercept, r, p_val, _ = stats.linregress(x_true, y_vals)
                r2   = r ** 2
                x_ln = np.linspace(x_true.min(), x_true.max(), 100)
                y_ln = slope * x_ln + intercept

                fig.add_trace(
                    go.Scatter(
                        x=x_ln, y=y_ln,
                        mode="lines",
                        legendgroup=ds_display,
                        showlegend=False,
                        line=dict(color=ds_color, dash="dash", width=1.5),
                        hovertemplate=(
                            f"<b>{ds_display}</b><br>"
                            f"R²={r2:.3f} | slope={slope:+.3f} | p={p_val:.3f}"
                            "<extra></extra>"
                        ),
                    ),
                    row=1, col=col_idx,
                )

                fig.add_annotation(
                    x=x_ln[-1],
                    y=float(np.clip(y_ln[-1], 0.03, 0.98)),
                    text=f"<b>R²={r2:.2f}</b>",
                    font=dict(size=9, color=ds_color),
                    showarrow=False,
                    xanchor="left",
                    xshift=5,
                    row=1, col=col_idx,
                )

        fig.update_xaxes(
            title_text=x_label,
            range=[x_start, None],
            zeroline=False,
            row=1, col=col_idx,
        )

    fig.update_yaxes(title_text="F1 score", range=[0, 1.05], row=1, col=1)
    fig.add_hline(y=0.5, line_dash="dot", line_color="lightgrey", line_width=1)

    jitter_note = (
        "<br><sup>x-axis jittered slightly to reveal overlapping points — hover for true values</sup>"
        if add_jitter else ""
    )
    fig.update_layout(
        title=dict(text=title + jitter_note, x=0.5, xanchor="center", font=dict(size=13)),
        legend=dict(title="Dataset", orientation="h", y=-0.20, x=0.5, xanchor="center"),
        height=500,
        width=550 if n_cols == 1 else 1050,
        template="plotly_white",
        margin=dict(t=90, b=110, l=65, r=50),
    )
    return fig


# ── Figure 1: growth rate vs. F1 (both panels) ────────────────────────────────
fig1 = _make_fig(
    x_col="growth_rate",
    x_label=f"WSF growth rate (ref year → {_late_year_label})",
    title="WSF growth rate vs. F1 score  |  dashed lines = OLS regression",
    x_start=0,
    add_jitter=False,
)
fig1.show()

# ── Figure 2: temporal gap vs. F1 (non-SpaceNet7 only) ────────────────────────
fig2 = _make_fig(
    x_col="temporal_gap_years",
    x_label="Temporal gap (years, ref year → mid-2025)",
    title="Temporal gap vs. F1 score  |  dashed lines = OLS regression",
    x_start=0,
    add_jitter=True,
    panels=[(False, "Non-SpaceNet7 cities")],
)
fig2.show()

# ── Save as interactive HTML ───────────────────────────────────────────────────
_out = PROJECT_ROOT / "outputs/figures"
_out.mkdir(parents=True, exist_ok=True)

fig1.write_html(str(_out / "growth_rate_vs_f1.html"))
fig2.write_html(str(_out / "temporal_gap_vs_f1.html"))
print("Saved → outputs/figures/growth_rate_vs_f1.html")
print("Saved → outputs/figures/temporal_gap_vs_f1.html")

Saved → outputs/figures/growth_rate_vs_f1.html
Saved → outputs/figures/temporal_gap_vs_f1.html


### Cell 7 — Correlation table (Pearson & Spearman)

In [44]:
from scipy import stats

X_VARS = {
    "growth_rate":        f"WSF growth rate (ref year → {_late_year_label})",
    "temporal_gap_years": "Temporal gap (years)",
}

# ── Restrict to cities covered by ALL datasets (fair comparison) ──────────────
datasets_present = sorted(merged_df["dataset_label"].unique())
city_coverage    = {
    ds: set(merged_df[merged_df["dataset_label"] == ds]["city"].unique())
    for ds in datasets_present
}
common_cities = set.intersection(*city_coverage.values())

print("Dataset coverage (all cities in merged_df):")
for ds, cities in sorted(city_coverage.items()):
    print(f"  {ds}: {len(cities)} cities")
print(f"\nCities present in ALL {len(datasets_present)} datasets: {len(common_cities)}")
print("Correlation analysis is restricted to this shared set.\n")

merged_common = merged_df[merged_df["city"].isin(common_cities)].copy()

# ── Correlation loop ──────────────────────────────────────────────────────────
# For temporal_gap_years, SpaceNet7 cities all share the same value (5.5 years)
# → constant input → NaN.  Skip that combination entirely.
GROUPS_BY_VAR = {
    "growth_rate":        [(True, "SpaceNet7"), (False, "Non-SpaceNet7")],
    "temporal_gap_years": [(False, "Non-SpaceNet7")],
}

corr_rows = []

for x_col, x_label in X_VARS.items():
    for ds_label in datasets_present:
        for is_sn7, group_name in GROUPS_BY_VAR[x_col]:
            subset = merged_common[
                (merged_common["dataset_label"] == ds_label) &
                (merged_common["is_spacenet7"]  == is_sn7)
            ].dropna(subset=[x_col, "f1"])

            n = len(subset)
            base = {
                "x_variable": x_col, "dataset": ds_label, "group": group_name, "n": n,
                "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_r": np.nan, "spearman_p": np.nan,
            }

            if n < 4:
                corr_rows.append({**base, "note": "n<4 — skipped"})
                continue

            x = subset[x_col].values.astype(float)
            y = subset["f1"].values.astype(float)

            if np.std(x) == 0:
                corr_rows.append({**base, "note": "constant x — skipped"})
                continue

            pr, pp = stats.pearsonr(x, y)
            sr, sp = stats.spearmanr(x, y)

            corr_rows.append({
                **base,
                "pearson_r":  round(float(pr), 3),
                "pearson_p":  round(float(pp), 4),
                "spearman_r": round(float(sr), 3),
                "spearman_p": round(float(sp), 4),
                "note": "significant (p<0.05)" if min(pp, sp) < 0.05 else "",
            })

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values(["x_variable", "dataset", "group"])
    .reset_index(drop=True)
)

# ── Print results ─────────────────────────────────────────────────────────────
for x_col, x_label in X_VARS.items():
    sub = corr_df[corr_df["x_variable"] == x_col].drop(columns="x_variable")
    print(f"=== Pearson & Spearman correlations: {x_label} vs. F1 ===")
    print(f"    (restricted to {len(common_cities)} cities shared across all datasets)\n")
    print(sub.to_string(index=False))

    sig = sub[sub["note"].str.contains("significant", na=False)]
    if not sig.empty:
        print(f"\n✓ Significant (p<0.05):")
        for _, r in sig.iterrows():
            print(f"  {r['dataset']} / {r['group']}: "
                  f"Pearson r={r.pearson_r} (p={r.pearson_p}), "
                  f"Spearman r={r.spearman_r} (p={r.spearman_p})")
    else:
        print("\n  No significant correlations found (p<0.05).")
    print()

out_path = PROJECT_ROOT / "outputs/wsf_growth_vs_f1_correlations.csv"
corr_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Dataset coverage (all cities in merged_df):
  gba: 105 cities
  globfp: 96 cities
  overture: 110 cities

Cities present in ALL 3 datasets: 96
Correlation analysis is restricted to this shared set.

=== Pearson & Spearman correlations: WSF growth rate (ref year → mid-2025) vs. F1 ===
    (restricted to 96 cities shared across all datasets)

 dataset         group  n  pearson_r  pearson_p  spearman_r  spearman_p                 note
     gba Non-SpaceNet7 48      0.262     0.0716       0.423      0.0028 significant (p<0.05)
     gba     SpaceNet7 48      0.029     0.8469      -0.009      0.9498                     
  globfp Non-SpaceNet7 48     -0.133     0.3684      -0.101      0.4963                     
  globfp     SpaceNet7 48     -0.296     0.0411      -0.180      0.2213 significant (p<0.05)
overture Non-SpaceNet7 48      0.028     0.8477      -0.006      0.9652                     
overture     SpaceNet7 48      0.033     0.8261       0.035      0.8135                     

✓ Sig

---

## Section 3: Per-City Temporally-Aligned WSF Validation

**Purpose:** Re-run WSF raster validation for each city using the `as_of_code` that best matches when that city's reference imagery was captured, rather than the global `as_of_code: 19` applied uniformly in the main pipeline.

**Why this matters:** The main pipeline compares every reference dataset against a WSF built-up mask that extends to Jan 2025 regardless of when the imagery was taken. A city with 2020 reference imagery is penalised for urban growth that occurred in 2021–2024 — those new buildings appear as False Positives. Aligning the WSF cutoff to the reference year isolates *spatial* accuracy from *temporal mismatch*.

**Method:** For each city, compute `wsf_code = round((ref_year − 2015) × 2)` and run `eval_wsf_for_city` at that code. The comparison cell then merges with Section 1 baseline results (as_of_code = 19) to show the F1 delta per city.

**Dependencies:** run Section 1 helpers cell (`eval_wsf_for_city`, `tracker`) and Section 2 cells 1–2 (`ref_year_map`, `ref_year_to_wsf_code`) before running this section. Section 1 sweep is optional but needed for the baseline comparison.

In [46]:
# ---- Section 3: per-city aligned validation loop ----
# Requires: ref_year_map + ref_year_to_wsf_code (Section 2 cells 1–2)
#           eval_wsf_for_city + tracker (Section 1)
#           sn7_cities (Section 2 cell 5) — for SpaceNet7 flag

import warnings as _w
_w.filterwarnings("always")
# filterwarnings("always") resets all filters, which re-enables the jupyter_client
# utcnow deprecation noise suppressed in the imports cell — re-suppress it here.
_w.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow",
    category=DeprecationWarning,
)

# sn7_cities is defined in Cell 5; fall back to empty set so this cell can
# run standalone (flag will show False for all cities in that case).
_sn7_lookup = sn7_cities if "sn7_cities" in dir() else set()
if not _sn7_lookup:
    print("⚠ sn7_cities not found — run Section 2 Cell 5 first for correct SpaceNet7 flags.")

aligned_rows    = []
skipped_aligned = []

for city, ref_year in ref_year_map.items():
    city_rows = tracker[tracker["Dataset code"] == city]
    if city_rows.empty:
        skipped_aligned.append(f"{city}: not in tracker")
        continue

    wsf_code = ref_year_to_wsf_code(ref_year)
    row      = city_rows.iloc[0]

    try:
        res = eval_wsf_for_city(city, row, as_of_code=wsf_code)
        if res is not None:
            res["ref_image_year"]   = ref_year
            res["wsf_code_aligned"] = wsf_code
            res["is_spacenet7"]     = city in _sn7_lookup
            aligned_rows.append(res)
            tag = " [SN7]" if res["is_spacenet7"] else ""
            print(
                f"  {city}{tag}"
                f"  (ref_year={ref_year}, code={wsf_code}): "
                f"F1={res['f1_area']:.3f}"
            )
        else:
            skipped_aligned.append(f"{city}: missing data")
    except Exception as exc:
        skipped_aligned.append(f"{city}: ERROR — {exc}")
        print(f"  {city}: ERROR — {exc}")

aligned_df = pd.DataFrame(aligned_rows)

n_sn7     = int(aligned_df["is_spacenet7"].sum())
n_non_sn7 = len(aligned_df) - n_sn7
print(
    f"\nAligned validation: {len(aligned_df)} cities  "
    f"({n_sn7} SpaceNet7, {n_non_sn7} non-SpaceNet7)  |  "
    f"{len(skipped_aligned)} skipped"
)
if skipped_aligned:
    print("Skipped:")
    for s in skipped_aligned:
        print(f"  {s}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

  ken-mukuru  (ref_year=2026, code=20): F1=0.851
  tto-la-brea  (ref_year=2026, code=20): F1=0.647
  phl-poblacion-sagua-anini-y-antique  (ref_year=2026, code=20): F1=0.830
  sle-freetown  (ref_year=2026, code=20): F1=0.814
  phl-visayas  (ref_year=2025, code=18): F1=0.726
  phl-juraojurao-anini-y-antique  (ref_year=2025, code=18): F1=0.804
  phl-catanduanes  (ref_year=2025, code=18): F1=0.752
  ken-kakuma  (ref_year=2025, code=18): F1=0.330
  sle-kolleh  (ref_year=2025, code=18): F1=0.902
  sle-cockle-bay-1  (ref_year=2025, code=18): F1=0.869
  sle-cockle-bay-2  (ref_year=2025, code=18): F1=0.803
  sle-kroo-bay-1  (ref_year=2025, code=18): F1=0.895
  ken-nairobi  (ref_year=2025, code=18): F1=0.586
  swz-nhlangano  (ref_year=2025, code=18): F1=0.648
  moz-djonasse  (ref_year=2025, code=18): F1=0.397
  moz-de-maio  (ref_year=2025, code=18): F1=0.595
  mmr-patheingyi-mandalay  (ref_year=2025, code=18): F1=0.597
  tto-sangre  (ref_year=2024, code=16): F1=0.462
  ken-kakuma-kalobeyei  (ref

/usr/local/lib/python3.12/dist-packages/rasterio/features.py:410: NotGeoreferencedWarning:

Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.



  lby-almarj  (ref_year=2024, code=16): F1=0.685
  mwi-mlowe  (ref_year=2023, code=14): F1=0.388
  lby-susah  (ref_year=2023, code=14): F1=0.682
  phl-bagamanoc  (ref_year=2023, code=14): F1=0.751
  uga-kanara  (ref_year=2023, code=14): F1=0.554
  uga-bugoye  (ref_year=2023, code=14): F1=0.624
  jpn-numakunai  (ref_year=2023, code=14): F1=0.693
  jpn-iwate  (ref_year=2023, code=14): F1=0.709
  jpn-izu-oshima  (ref_year=2023, code=14): F1=0.717
  bra-nova-sussuarana  (ref_year=2022, code=12): F1=0.132
  gha-nawuni  (ref_year=2022, code=12): F1=0.662
  phl-pasig  (ref_year=2022, code=12): F1=0.853
  ukr-pulyny  (ref_year=2022, code=12): F1=0.643
  tjk-nomandiyon  (ref_year=2022, code=12): F1=0.666
  tjk-artuch  (ref_year=2022, code=12): F1=0.779
  phl-viga  (ref_year=2021, code=10): F1=0.793
  tjk-tavishi-bolo  (ref_year=2021, code=10): F1=0.645
  gha-dansoman  (ref_year=2021, code=10): F1=0.727
  jam-kingston  (ref_year=2021, code=10): F1=0.642
  ner-niame  (ref_year=2021, code=10): F1=

/usr/local/lib/python3.12/dist-packages/rasterio/features.py:410: NotGeoreferencedWarning:

Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.



  mwi-lilongwe  (ref_year=2021, code=10): F1=0.561
  blz-burrell-boom  (ref_year=2020, code=8): F1=0.373
  sxm-sint-maarten  (ref_year=2020, code=8): F1=0.613
  uga-nakamiro  (ref_year=2020, code=8): F1=0.793
  bgd-rohingya  (ref_year=2020, code=8): F1=0.669
  phl-barangay  (ref_year=2019, code=6): F1=0.731
  jpn-ashiya-hama  (ref_year=2019, code=6): F1=0.752
  jpn-hiroshima  (ref_year=2019, code=6): F1=0.748
  ton-tokomololo  (ref_year=2018, code=4): F1=0.582
  ton-nukualofa  (ref_year=2018, code=4): F1=0.627
  ton-sopu  (ref_year=2018, code=4): F1=0.644
  ton-tatakamotonga  (ref_year=2018, code=4): F1=0.473
  maf-saint-martin  (ref_year=2017, code=2): F1=0.600
  afg-mahmud-e-raqi  (ref_year=2020, code=8): F1=0.626
  afg-mazar-e-sharif  (ref_year=2020, code=8): F1=0.669
  afg-qala-e-naw  (ref_year=2020, code=8): F1=0.614
  afg-zaranj  (ref_year=2020, code=8): F1=0.661
  alg-tindouf [SN7]  (ref_year=2020, code=8): F1=0.538
  are-abudhabi [SN7]  (ref_year=2020, code=8): F1=0.545
  aus-m

/usr/local/lib/python3.12/dist-packages/rasterio/features.py:410: NotGeoreferencedWarning:

Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.



  lca-saint-lucia  (ref_year=2020, code=8): F1=0.587
  mex-mexicocity [SN7]  (ref_year=2020, code=8): F1=0.740
  nld-rotterdam [SN7]  (ref_year=2020, code=8): F1=0.370
  pan-panamacity [SN7]  (ref_year=2020, code=8): F1=0.628
  per-cusco [SN7]  (ref_year=2020, code=8): F1=0.652
  phl-angelescity [SN7]  (ref_year=2020, code=8): F1=0.635
  rou-bucharest [SN7]  (ref_year=2020, code=8): F1=0.503
  rus-astrakhan [SN7]  (ref_year=2020, code=8): F1=0.431
  sau-riyadh [SN7]  (ref_year=2020, code=8): F1=0.668
  sdn-khartoum [SN7]  (ref_year=2020, code=8): F1=0.507
  sen-dakar [SN7]  (ref_year=2020, code=8): F1=0.652
  usa-allentown [SN7]  (ref_year=2020, code=8): F1=0.675
  usa-atlanta [SN7]  (ref_year=2020, code=8): F1=0.677
  usa-bentonville [SN7]  (ref_year=2020, code=8): F1=0.620
  usa-boise [SN7]  (ref_year=2020, code=8): F1=0.543
  usa-brentwood [SN7]  (ref_year=2020, code=8): F1=0.670
  usa-gonzales [SN7]  (ref_year=2020, code=8): F1=0.595
  usa-lasvegas [SN7]  (ref_year=2020, code=8): F

In [48]:
# ---- Section 3: compare aligned vs. pipeline baseline ----

BASELINE_CODE_S3 = 19

if (
    "sensitivity_df" in dir()
    and not sensitivity_df.empty
    and BASELINE_CODE_S3 in sensitivity_df["as_of_code"].values
):
    baseline_city = (
        sensitivity_df[sensitivity_df["as_of_code"] == BASELINE_CODE_S3]
        [["city", "f1_area", "precision_area", "recall_area"]]
        .rename(columns={
            "f1_area":        "f1_baseline",
            "precision_area": "precision_baseline",
            "recall_area":    "recall_baseline",
        })
    )
    comparison_df = aligned_df.merge(baseline_city, on="city", how="left")
    comparison_df["f1_delta"]        = comparison_df["f1_area"]        - comparison_df["f1_baseline"]
    comparison_df["precision_delta"] = comparison_df["precision_area"] - comparison_df["precision_baseline"]
    comparison_df["recall_delta"]    = comparison_df["recall_area"]    - comparison_df["recall_baseline"]
    has_baseline = True
else:
    comparison_df = aligned_df.copy()
    has_baseline  = False
    print("Note: sensitivity_df not available — run Section 1 first for baseline comparison.")

# ── Summary: overall and by SpaceNet7 group ───────────────────────────────────
def _group_summary(df, label):
    lines = [f"  {label}  (n={len(df)})"]
    lines.append(
        f"    Aligned :  F1={df['f1_area'].mean():.4f}  "
        f"precision={df['precision_area'].mean():.4f}  "
        f"recall={df['recall_area'].mean():.4f}"
    )
    if has_baseline:
        lines.append(
            f"    Baseline:  F1={df['f1_baseline'].mean():.4f}  "
            f"precision={df['precision_baseline'].mean():.4f}  "
            f"recall={df['recall_baseline'].mean():.4f}"
        )
        f1d  = df["f1_delta"].dropna()
        prd  = df["precision_delta"].dropna()
        rcd  = df["recall_delta"].dropna()
        lines.append(
            f"    Δ (aligned − baseline): "
            f"F1={f1d.mean():+.4f}  "
            f"precision={prd.mean():+.4f}  "
            f"recall={rcd.mean():+.4f}  "
            f"| F1 improved(>+0.01)={( f1d >  0.01).sum()}  "
            f"degraded(<-0.01)={( f1d < -0.01).sum()}"
        )
    return "\n".join(lines)

print("=== Per-city temporally-aligned WSF validation ===\n")
print(_group_summary(comparison_df,                                 "All cities"))
print()
print(_group_summary(comparison_df[ comparison_df["is_spacenet7"]], "SpaceNet7"))
print()
print(_group_summary(comparison_df[~comparison_df["is_spacenet7"]], "Non-SpaceNet7"))

if has_baseline:
    print(
        f"\nNote: positive precision Δ = fewer false positives (growth buildings removed)."
        f"\n      negative recall Δ    = expected if aligned WSF covers less built area."
    )

# ── Full table ────────────────────────────────────────────────────────────────
cols_show = [
    "city", "is_spacenet7", "ref_image_year", "wsf_code_aligned",
    "f1_area", "precision_area", "recall_area",
]
if has_baseline:
    cols_show += [
        "f1_baseline",        "f1_delta",
        "precision_baseline", "precision_delta",
        "recall_baseline",    "recall_delta",
    ]

sort_col = "f1_delta" if has_baseline else "f1_area"
display(
    comparison_df[cols_show]
    .sort_values(sort_col, ascending=True)
    .reset_index(drop=True)
)

out_path = PROJECT_ROOT / "outputs/wsf_aligned_validation.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

=== Per-city temporally-aligned WSF validation ===

  All cities  (n=120)
    Aligned :  F1=0.6261  precision=0.4750  recall=0.9641
    Baseline:  F1=0.5983  precision=0.4424  recall=0.9917
    Δ (aligned − baseline): F1=+0.0278  precision=+0.0326  recall=-0.0276  | F1 improved(>+0.01)=58  degraded(<-0.01)=2

  SpaceNet7  (n=49)
    Aligned :  F1=0.6086  precision=0.4545  recall=0.9472
    Baseline:  F1=0.5475  precision=0.3850  recall=1.0000
    Δ (aligned − baseline): F1=+0.0611  precision=+0.0694  recall=-0.0528  | F1 improved(>+0.01)=44  degraded(<-0.01)=1

  Non-SpaceNet7  (n=71)
    Aligned :  F1=0.6381  precision=0.4891  recall=0.9757
    Baseline:  F1=0.6333  precision=0.4820  recall=0.9859
    Δ (aligned − baseline): F1=+0.0048  precision=+0.0071  recall=-0.0102  | F1 improved(>+0.01)=14  degraded(<-0.01)=1

Note: positive precision Δ = fewer false positives (growth buildings removed).
      negative recall Δ    = expected if aligned WSF covers less built area.


,city,is_spacenet7,ref_image_year,wsf_code_aligned,f1_area,precision_area,recall_area,f1_baseline,f1_delta,precision_baseline,precision_delta,recall_baseline,recall_delta
0,usa-permianbasin,False,2020,8,0.3896,0.2459,0.9375,0.4051,-0.0155,0.2540,-0.0081,1.0,-0.0625
1,yem-dhamar,True,2020,8,0.6186,0.5132,0.7784,0.6308,-0.0122,0.4607,0.0525,1.0,-0.2216
2,afg-qala-e-naw,False,2020,8,0.6142,0.4621,0.9159,0.6234,-0.0092,0.4529,0.0092,1.0,-0.0841
3,phl-visayas,False,2025,18,0.7258,0.5759,0.9813,0.7329,-0.0071,0.5784,-0.0025,1.0,-0.0187
4,jpn-ashiya-hama,False,2019,6,0.7520,0.6135,0.9714,0.7569,-0.0049,0.6089,0.0046,1.0,-0.0286
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,usa-bentonville,True,2020,8,0.6204,0.4548,0.9758,0.4813,0.1391,0.3169,0.1379,1.0,-0.0242
116,kor-sejong,True,2020,8,0.4509,0.3104,0.8240,0.2798,0.1711,0.1627,0.1477,1.0,-0.1760
117,ind-tada,True,2020,8,0.6334,0.4875,0.9038,0.4519,0.1815,0.2919,0.1956,1.0,-0.0962
118,aus-melbourne,True,2020,8,0.6137,0.4888,0.8244,0.3751,0.2386,0.2308,0.2580,1.0,-0.1756



Saved → /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/wsf_aligned_validation.csv


In [49]:
# ── Summary of findings across all three sections ────────────────────────────
#
# Reads from: summary (S1), corr_df / common_cities (S2 cell 7),
#             comparison_df / has_baseline (S3).
# Run all sections before executing this cell.

import textwrap

_sep  = "=" * 70
_sep2 = "-" * 70

def _wrap(text, indent=4):
    return textwrap.fill(text, width=70, initial_indent=" " * indent,
                         subsequent_indent=" " * indent)

print(_sep)
print("FINDINGS SUMMARY — WSF Tracker year-code sensitivity analysis")
print(_sep)

# ── Section 1 ────────────────────────────────────────────────────────────────
print("\n[1] Year-code sensitivity (Section 1)\n")

if "summary" in dir() and not summary.empty:
    f1_min  = summary["mean_f1"].min()
    f1_max  = summary["mean_f1"].max()
    f1_rng  = f1_max - f1_min
    bl_row  = summary[summary["as_of_code"] == BASELINE_CODE]
    f1_bl   = float(bl_row["mean_f1"].iloc[0]) if not bl_row.empty else float("nan")
    n_flag  = int(summary["flagged"].sum()) if "flagged" in summary.columns else 0

    yr_min  = summary["as_of_year"].min()
    yr_max  = summary["as_of_year"].max()

    print(_wrap(
        f"Across {len(summary)} year-code cutoffs tested ({yr_min}–{yr_max}), "
        f"mean city-level F1 ranged from {f1_min:.3f} to {f1_max:.3f} "
        f"(spread: {f1_rng:.3f})."
    ))
    print(_wrap(
        f"The pipeline baseline (as_of_code={BASELINE_CODE}, ≈mid-2025) "
        f"gives mean F1={f1_bl:.3f}."
    ))
    if n_flag == 0:
        print(_wrap(
            "No cutoff differs from the baseline by more than 0.05 F1, "
            "indicating that the choice of year-code has limited sensitivity "
            "on aggregate accuracy."
        ))
    else:
        print(_wrap(
            f"{n_flag} cutoff(s) differ from the baseline by >0.05 F1 — "
            "temporal choice has a measurable effect at those cutoffs."
        ))
else:
    print("    Section 1 results not available (sensitivity_df not in memory).")

# ── Section 2 ────────────────────────────────────────────────────────────────
print(f"\n{_sep2}")
print("\n[2] Urban growth rate & temporal gap vs. F1 (Section 2)\n")

if "corr_df" in dir() and not corr_df.empty:
    n_common = len(common_cities) if "common_cities" in dir() else "?"
    print(_wrap(
        f"Correlations were computed on the {n_common} cities covered by all "
        f"datasets, split by SpaceNet7 / non-SpaceNet7."
    ))
    print()

    # Growth rate — significant rows
    gr_sig = corr_df[
        (corr_df["x_variable"] == "growth_rate") &
        (corr_df["note"].str.contains("significant", na=False))
    ]
    if not gr_sig.empty:
        print("  Growth rate vs. F1 — significant correlations:")
        for _, r in gr_sig.iterrows():
            direction = "positive" if r.pearson_r > 0 else "negative"
            print(_wrap(
                f"{r['dataset']} / {r['group']}: Spearman r={r.spearman_r:+.3f} "
                f"(p={r.spearman_p:.4f}), Pearson r={r.pearson_r:+.3f} "
                f"(p={r.pearson_p:.4f}) — {direction} association.",
                indent=6,
            ))
    else:
        print(_wrap(
            "Growth rate vs. F1: no significant correlations (p<0.05) "
            "found in any dataset / group combination."
        ))

    print()

    # Temporal gap — significant rows (non-SN7 only)
    tg_sig = corr_df[
        (corr_df["x_variable"] == "temporal_gap_years") &
        (corr_df["note"].str.contains("significant", na=False))
    ]
    if not tg_sig.empty:
        print("  Temporal gap vs. F1 — significant correlations (non-SpaceNet7):")
        for _, r in tg_sig.iterrows():
            direction = "positive" if r.pearson_r > 0 else "negative"
            print(_wrap(
                f"{r['dataset']} / {r['group']}: Spearman r={r.spearman_r:+.3f} "
                f"(p={r.spearman_p:.4f}), Pearson r={r.pearson_r:+.3f} "
                f"(p={r.pearson_p:.4f}) — {direction} association.",
                indent=6,
            ))
        print(_wrap(
            "SpaceNet7 cities are excluded from temporal gap correlations "
            "because they share a fixed assumed reference year (2020), "
            "giving a constant temporal gap of 5.5 years.",
            indent=4,
        ))
    else:
        print(_wrap(
            "Temporal gap vs. F1: no significant correlations found "
            "for non-SpaceNet7 cities."
        ))
else:
    print("    Section 2 correlation results not available (corr_df not in memory).")

# ── Section 3 ────────────────────────────────────────────────────────────────
print(f"\n{_sep2}")
print("\n[3] Temporally-aligned validation (Section 3)\n")

if "comparison_df" in dir() and not comparison_df.empty and has_baseline:
    for mask, label in [
        (slice(None),                              "All cities"),
        (comparison_df["is_spacenet7"] == True,   "SpaceNet7"),
        (comparison_df["is_spacenet7"] == False,  "Non-SpaceNet7"),
    ]:
        sub = comparison_df.loc[mask] if not isinstance(mask, slice) else comparison_df
        n   = len(sub)
        if n == 0:
            continue

        f1d  = sub["f1_delta"].dropna()
        prd  = sub["precision_delta"].dropna()
        rcd  = sub["recall_delta"].dropna()
        imp  = (f1d >  0.01).sum()
        deg  = (f1d < -0.01).sum()

        print(f"  {label} (n={n}):")
        print(_wrap(
            f"Aligning WSF to each city's reference image year changes mean F1 "
            f"by {f1d.mean():+.4f} ({imp} cities improve >0.01, {deg} degrade >0.01). "
            f"Precision Δ = {prd.mean():+.4f}, recall Δ = {rcd.mean():+.4f}.",
            indent=6,
        ))
        print()

    # Interpret precision / recall direction
    overall_prd = comparison_df["precision_delta"].dropna().mean()
    overall_rcd = comparison_df["recall_delta"].dropna().mean()
    if overall_prd > 0 and overall_rcd < 0:
        print(_wrap(
            "The pattern is consistent with the expected temporal-mismatch effect: "
            "using the reference-year WSF mask removes post-acquisition growth pixels, "
            "reducing false positives (higher precision) while also covering less "
            "built-up area, so some true positives are lost (lower recall)."
        ))
    elif overall_prd > 0 and overall_rcd >= 0:
        print(_wrap(
            "Precision improves without a recall penalty, suggesting that temporal "
            "alignment removes spurious false positives without losing true detections."
        ))
    else:
        print(_wrap(
            "The precision/recall pattern does not match the simple temporal-mismatch "
            "hypothesis — inspect per-city deltas for outliers."
        ))
elif "comparison_df" in dir() and not comparison_df.empty:
    # No baseline available — just print aligned results
    sub = comparison_df
    print(_wrap(
        f"Aligned validation completed for {len(sub)} cities "
        f"(mean F1={sub['f1_area'].mean():.4f}, "
        f"precision={sub['precision_area'].mean():.4f}, "
        f"recall={sub['recall_area'].mean():.4f}). "
        "Run Section 1 first to enable baseline comparison."
    ))
else:
    print("    Section 3 results not available (comparison_df not in memory).")

print(f"\n{_sep}")

FINDINGS SUMMARY — WSF Tracker year-code sensitivity analysis

[1] Year-code sensitivity (Section 1)

    Across 9 year-code cutoffs tested (2019.0–2026.0), mean city-level
    F1 ranged from 0.597 to 0.617 (spread: 0.020).
    The pipeline baseline (as_of_code=19, ≈mid-2025) gives mean
    F1=0.597.
    No cutoff differs from the baseline by more than 0.05 F1,
    indicating that the choice of year-code has limited sensitivity on
    aggregate accuracy.

----------------------------------------------------------------------

[2] Urban growth rate & temporal gap vs. F1 (Section 2)

    Correlations were computed on the 96 cities covered by all
    datasets, split by SpaceNet7 / non-SpaceNet7.

  Growth rate vs. F1 — significant correlations:
      gba / Non-SpaceNet7: Spearman r=+0.423 (p=0.0028), Pearson
      r=+0.262 (p=0.0716) — positive association.
      globfp / SpaceNet7: Spearman r=-0.180 (p=0.2213), Pearson
      r=-0.296 (p=0.0411) — negative association.

  Temporal gap vs.